# Stock Price Prediction

### Libraries 

In [7]:
import pandas as pd
import numpy as np
import os

#### Importing data

In [8]:
companies = ['Amazon', 'Apple', 'Google', 'Microsoft', 'Netflix']
data_dir = '../Data/stock_data/'

data = {}
for name in companies:
    file_path = os.path.join(data_dir, f"{name}.csv")
    df = pd.read_csv(file_path, parse_dates= ['Date'])
    df = df.sort_values('Date').reset_index(drop= True)
    df = df.rename(columns = {"Adj Close" : "Adj_Close"})
    df['Company'] = name
    data[name] = df

### EDA & Feature Engineering 

In [11]:
# daily return column
for name, df in data.items():
    df['Return'] = df['Adj_Close'].pct_change()
    data[name] = df

# label column 
for name, df in data.items():
    df['Label'] = (df['Adj_Close'].shift(-1) > df['Adj_Close']).astype(int)
    data[name] = df

# rolling moving average of 5/ 10/ 20 - days columns
for name, df in data.items():
    df['MA_5'] = df['Adj_Close'].rolling(window = 5).mean()
    df['MA_10'] = df['Adj_Close'].rolling(window = 10).mean()
    df['MA_20'] = df['Adj_Close'].rolling(window = 20).mean()
    data[name] = df

# rsi column
for name, df in data.items():
    change = df['Adj_Close'] - df['Adj_Close'].shift(1)
    df['gain'] = change.where(change > 0, 0)
    df['loss'] = -change.where(change < 0, 0)
    avg_gain = df['gain'].rolling(window = 14).mean()
    avg_loss = df['loss'].rolling(window = 14).mean()
    rs = avg_gain / avg_loss
    df['rsi'] = 100 - (100 / (1 + rs))
    data[name] = df
